In [1]:
import pandas as pd


In [2]:
train = pd.read_csv('dataset//train_tokenized.csv')
test = pd.read_csv('dataset//test_tokenized.csv')

In [3]:
from tokenizer import WordLevelTokenizer

english_tokenizer = WordLevelTokenizer().load('english_tokenizer.json')
tamil_tokenizer = WordLevelTokenizer().load('tamil_tokenizer.json')

In [4]:
tamil_tokenizer.vocab_size()

15529

In [5]:
from data_loader import TranslationDataset, collate_fn
from torch.utils.data import DataLoader

In [13]:
train_dataset = TranslationDataset(train)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

test_dataset = TranslationDataset(test)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)


In [14]:
for batch in test_dataloader:
    print(english_tokenizer.decode(batch["english_token"][0].tolist()))        # (B, src_len)
    print(tamil_tokenizer.decode(batch["tamil_token"][0].tolist()))          # (B, tgt_len)
    print(tamil_tokenizer.decode(batch["tamil_target"][0].tolist()))         # (B, tgt_len)

    print(batch["encoder_mask"][0])   # (B, 1, 1, src_len)
    print(batch["decoder_mask"][0])   # (B, 1, tgt_len, tgt_len)
    break

I rode with her as far as the station
நான் அவளுடன் ஸ்டேஷன் வரை சவாரி செய்தேன்
நான் அவளுடன் ஸ்டேஷன் வரை சவாரி செய்தேன்
tensor([[[True, True, True, True, True, True, True, True, True, True, True]]])
tensor([[[ True, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True, False, False],
         [ True,  True,  True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True,  True,  True]]])


In [15]:
from model import Transformer

model = Transformer( src_vocab = 8100, tgt_vocab = 15529, d_model = 64, n_heads = 8, d_ff = 512, n_enc = 8, n_dec = 8, dropout=0.1)


In [16]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters()) 

total_params = count_parameters(model)
print(f"Total model parameters: {total_params}")

Total model parameters: 3983913


In [18]:
from training import train_model

max_len = 50
device = "cuda"
epochs = 17

train_model(train_dataloader = train_dataloader, test_dataloader = test_dataloader, model = model, english_tokenizer = english_tokenizer,
            tamil_tokenizer = tamil_tokenizer, max_len = max_len, device = device, epochs = epochs, lr = 1e-4)

Using device: cuda
Device name: NVIDIA GeForce GTX 1060 6GB
Device memory: 5.999755859375 GB
Checkpoint found. Loading model and optimizer state...
Model loaded from checkpoint. Resuming from epoch 84.


Epoch 84: 100%|██████████| 156/156 [00:18<00:00,  8.28it/s, loss=3.4838]


     SOURCE:You should have told me the truth
     TARGET:நீங்கள் என்னிடம் உண்மையைச் சொல்லியிருக்க வேண்டும்
  PREDICTED:நீங்கள் என்னிடம் உண்மையைச் சொல்லியிருக்க வேண்டும்
--------------------------------------------------------------------------------


Epoch 85: 100%|██████████| 156/156 [00:17<00:00,  8.71it/s, loss=3.4842]


     SOURCE:The conference is to be held in Tokyo
     TARGET:இந்த மாநாடு டோக்கியோவில் நடைபெற உள்ளது
  PREDICTED:மாநாடு முடிவடையும் தருவாயில் உள்ளது
--------------------------------------------------------------------------------


...

     SOURCE:How wonderful a time we have had
     TARGET:எவ்வளவு அற்புதமான காலத்தை நாம் பெற்றிருக்கிறோம்
  PREDICTED:நாங்கள் எவ்வளவு நேரம் ஒரு நேரம் உள்ளது
--------------------------------------------------------------------------------


Epoch 99: 100%|██████████| 156/156 [00:18<00:00,  8.33it/s, loss=3.0232]


     SOURCE:The workers complained when their working hours were extended
     TARGET:வேலை நேரம் நீட்டிக்கப்பட்டது குறித்து தொழிலாளர்கள் புகார் தெரிவித்தனர்
  PREDICTED:படம் மணி நேரம் வேலை செய்பவர்கள் தொடங்கியது
--------------------------------------------------------------------------------


Epoch 100: 100%|██████████| 156/156 [00:18<00:00,  8.43it/s, loss=2.9788]


     SOURCE:Now I have to leave theyre calling for my flight
     TARGET:இப்போது நான் புறப்பட வேண்டும் அவர்கள் எனது விமானத்திற்கு அழைக்கிறார்கள்
  PREDICTED:நான் வங்கியில் என் நாட்கள் போய்விட்டன என கேட்டேன்
--------------------------------------------------------------------------------


In [ ]:

from testing import translate_loop

translate_loop(
    model,
    english_tokenizer,
    tamil_tokenizer,
    device
)

AttributeError: 'list' object has no attribute 'ids'

In [ ]:
checkpoint_path = "assets/translation_model_v1.pth"
checkpoint = torch.load(checkpoint_path ,map_location = "cuda")
model.load_state_dict(checkpoint['model_state_dict'])